
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Applying Agent Evaluation

## Overview

This lab provides hands-on experience with MLflow's evaluation framework for generative AI agents. You will work with the NYC Taxi dataset to build evaluation workflows using both built-in judges (correctness, safety) and custom guideline judges.

In this lab, you will create evaluation datasets, configure multiple types of judges, and analyze evaluation results to understand agent performance. The lab emphasizes practical implementation of automated evaluation workflows that can be applied to real-world AI applications.

## Learning Objectives

By the end of this lab, you will be able to:

- **Configure and execute** MLflow's built-in judges for correctness and safety assessment
- **Implement guideline judges** for custom business rule evaluation using natural language criteria
- **Create evaluation datasets** with proper structure for different judge types
- **Analyze evaluation results** using MLflow's tracing capabilities and result dataframes
- **Apply best practices** for separating evaluation models and managing configuration files


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;"> This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">That you might need to rerun the cells in the notebook if you receive an <strong>ValueError</strong> or <strong>MLflowException</strong> error regarding a datatype that was generated as a part of the agent's trace.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-4

## B. Test Agent Functions

### B1. Test Agent Functions

Let's test the agent's capabilities by calling it with different types of queries to understand how it uses the available functions.

In [0]:
def agent_payload(question: str):
    return [{"input": [{"role": "user", "content": question}]}]

In [0]:
## Test the agent with a query about trip information
## Use a query that would trigger the get_trip_info function
test_query_1 = "What's the average trip distance for trips from zip code 10001 to 10002?"
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task B1: Test Trip Info Query
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
test_query_1 = "What's the average trip distance for trips from zip code 10001 to 10002?"
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")
```

</div>
</details>

In [0]:
## Test the agent with a fare calculation query
## Use a query that would trigger the calculate_fare_estimate function
test_query_2 = "How much would it cost approximately to travel from 10001 to 10002?"
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task B1: Test Fare Calculation Query
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
test_query_2 = "What would be the estimated fare for a 5.2 mile trip with a base fare of $3.50?"
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")
```

</div>
</details>

## C. Built-In Judges Implementation

### C1. Create Evaluation Dataset for Built-In Judges

Create an evaluation dataset with two inputs, expected facts, and expected responses for testing correctness and safety judges. Part of the evaluation dataset has been filled out for you.

**Hint:** If you get stuck, you can view a sample evaluation dataset in your volume at `/Volumes/{catalog_name}/{schema_name}/agent_vol/nyc_taxi_eval.json`.

In [0]:
## Create an evaluation dataset for built-in judges
## Include inputs and expectations with expected_facts for each query
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What's the average trip distance for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average distance information, which is 2.32"
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "How much would it cost approximately to travel from 10001 to 10002?"

        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average fare information",
        "Triggers tool calling to calculate the average fare"
      ]
    }
  }
]

print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C1: Create Evaluation Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average fare information"
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "How many trips were taken from zip code 10003 to 10004?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10003",
        "Dropoff zip code is 10004",
        "Response includes trip count",
        "Response includes 'Are there any additional questions I can help with?'"
      ]
    }
  }
]
print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)
```

</div>
</details>

### C2. Configure Built-In Judges

Set up the Correctness and Safety judges using the configured endpoints.

In [0]:
## Import and configure the built-in judges
from mlflow.genai.scorers import Correctness, Safety

## Create correctness judge instance
correctness_eval = Correctness(
    model=correctness_eval_endpoint
)

## Create safety judge instance
safety_eval = Safety(
    model=safety_endpoint
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C2: Configure Built-In Judges
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import Correctness, Safety
# Create correctness judge instance
correctness_eval = Correctness(
    model=correctness_eval_endpoint
)
# Create safety judge instance
safety_eval = Safety(
    model=safety_endpoint
)
```

</div>
</details>

### C3. Execute Built-In Judge Evaluation

Run the evaluation using both correctness and safety judges simultaneously. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with both built-in judges
builtin_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input":input}),
    scorers=[correctness_eval, safety_eval]
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C3: Execute Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
builtin_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, safety_eval]
)
```

</div>
</details>

### C4. Analyze Built-In Judge Results

Examine the evaluation results and understand the scoring rationale by accessing `builtin_results.metrics` and `builtin_results.result_df`.

In [0]:
## Display the evaluation results and metrics
print(f"Run ID: {builtin_results.run_id}")
print(f"Aggregated Metrics: {builtin_results.metrics}")
print("\nDetailed Results:")
display(builtin_results.result_df)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C4: Analyze Results
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
print(f"Run ID: {builtin_results.run_id}")
print(f"Aggregated Metrics: {builtin_results.metrics}")
print("\nDetailed Results:")
display(builtin_results.result_df)
```

</div>
</details>

### C5. Update and Iterate (Optional)

Did your agent pass your expectations? If not, try updating the system prompt in `Includes/artifacts/configs/nyc_taxi_agent_config.yaml` so that the agent aligns with your evaluation expectations. Iterate until all evaluations **Pass**.

For example, the default system prompt is

```
You are an expert in answering questions regarding NYC Taxi data.
```

Which isn't very descriptive if, for example, we pass the question _What is the average fare for trips from zip code 10001 to 10002?_ and expect the following in our output
- Pickup zip code is 10001,
- Dropoff zip code is 10002,
- Response includes average fare information,
- Response includes 'Are there any additional questions I can help with?'

then we might update our system prompt with _You are an expert in answering questions regarding NYC Taxi data. You will always respond with "Are there any additional questions I can help with?"_ (see screenshot below).

![update_system_prompt.png](../Includes/images/applying agent eval/update_system_prompt.png "update_system_prompt.png")

After updating the prompt, you can redeploy your agent with the following code snippet.

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">CODE</span> Redeploy Agent with Updated Prompt
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow
import yaml
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

# Point to the rendered config file created by the setup orchestrator
CONFIG_YAML = "../Includes/artifacts/configs/nyc_taxi_agent_config.yaml"

# Load config from YAML
with open(CONFIG_YAML, "r") as f:
    cfg = yaml.safe_load(f)

# Pull values from YAML
CATALOG = cfg["CATALOG_NAME"]
SCHEMA = cfg["SCHEMA_NAME"]
MODEL_BASE_NAME = cfg.get("MODEL_NAME", "nyc_taxi")
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}"
LLM_ENDPOINT = cfg["LLM_ENDPOINT_NAME"]

# Build tool list from config
tool1 = cfg.get("TOOL1")
tool2 = cfg.get("TOOL2")
tool_names = [f"{CATALOG}.{SCHEMA}.{t}" for t in [tool1, tool2] if t]

# Build MLflow resources
resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT)] + [
    DatabricksFunction(function_name=t) for t in tool_names
]

# Agent source file (created by the setup orchestrator)
AGENT_PY = "../Includes/artifacts/nyc_taxi_agent.py"
ALIAS = "challenger"

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(tags={
    "training_type": "agent_eval_training",
    "model": LLM_ENDPOINT,
    "agent_type": "TOOL-CALLING",
}):
    logged = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=AGENT_PY,
        model_config=CONFIG_YAML,
        artifacts={"agent_config": CONFIG_YAML},
        input_example={"input": [{"role": "user", "content": "hello"}]},
        pip_requirements=["databricks-openai", "backoff", "pyyaml"],
        resources=resources,
        registered_model_name=UC_MODEL_NAME,
    )

new_version = logged.registered_model_version
print("New UC version:", new_version)

from mlflow.tracking import MlflowClient
MlflowClient().set_registered_model_alias(
    name=UC_MODEL_NAME, alias=ALIAS, version=new_version
)

agent = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}@{ALIAS}")
```

</div>
</details>

You can now go back to sections **C3** and **C4** and re-evaluate.

## D. Guideline Judges Implementation

### D1. Create Global Guidelines Dataset

Create an evaluation dataset for testing global guidelines that apply uniformly to all responses. Since the `Guidelines` judge defines rules at the judge level (not per-row), the dataset only needs `inputs`.

In [0]:
## Create evaluation dataset for global guidelines
eval_dataset_guidelines = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        },
        {
          "role": "user",
          "content": "How much would it cost approximately to travel from 10001 to 10002?"
        }
      ]
    }
  }
]
print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D1: Create Guidelines Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
eval_dataset_guidelines = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    }
  }
]
print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)
```

</div>
</details>

### D2. Implement Global Guidelines Judge

Create a global guidelines judge that ensures responses are professional and include specific formatting requirements.

In [0]:
## Import and configure the Guidelines judge
from mlflow.genai.scorers import Guidelines

## Create a global guideline for professional responses
professional_guideline = Guidelines(
    name="professional",
    guidelines=[
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ],
    model_name=guidelines_endpoint
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D2: Implement Guidelines Judge
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import Guidelines
# Create a global guideline for professional responses
professional_guideline = Guidelines(
    name="professional_response",
    guidelines=[
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ],
    model_name=guidelines_endpoint
)
```

</div>
</details>

### D3. Execute Global Guidelines Evaluation

Run the evaluation using the global guidelines judge. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with global guidelines
guidelines_results = mlflow.genai.evaluate(
    data=eval_dataset_guidelines,
    predict_fn=lambda input: agent.predict({"input":input}),
    scorers=[professional_guideline]
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D3: Execute Guidelines Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
guidelines_results = mlflow.genai.evaluate(
    data=eval_dataset_guidelines,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[professional_guideline]
)
```

</div>
</details>

## E. Comprehensive Evaluation Analysis

### E1. Compare Evaluation Results

Analyze and compare results from different judge types to understand their strengths and use cases.

In [0]:
## Display results from all evaluation runs for comparison
print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task E1: Compare Results
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {builtin_results.run_id}")
print(f"Metrics: {builtin_results.metrics}")
display(builtin_results.result_df)
print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {guidelines_results.run_id}")
print(f"Metrics: {guidelines_results.metrics}")
display(guidelines_results.result_df)
```

</div>
</details>

### E2. Create Comprehensive Evaluation

Combine multiple judge types in a single evaluation run to get comprehensive quality assessment.

In [0]:
## Create a comprehensive evaluation using multiple judge types
## Use the built-in dataset and combine correctness with global guidelines
comprehensive_results = mlflow.genai.evaluate(
    data=<FILL_IN>,
    predict_fn=<FILL_IN>,
    scorers=<FILL_IN>
)

print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task E2: Comprehensive Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
comprehensive_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, professional_guideline]
)
print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {comprehensive_results.run_id}")
print(f"Metrics: {comprehensive_results.metrics}")
display(comprehensive_results.result_df)
```

</div>
</details>

## F. Conclusion

In this lab, you have successfully implemented a comprehensive evaluation framework for AI agents using MLflow's built-in and guideline judges. You learned how to:

- Configure and execute multiple types of judges for different evaluation needs
- Create properly structured evaluation datasets for various judge types
- Analyze evaluation results using MLflow's tracing and result analysis capabilities
- Apply best practices for evaluation configuration and model management

These evaluation techniques provide the foundation for ensuring your AI agents meet quality standards and perform reliably in production environments. The combination of objective built-in judges and flexible guideline judges enables comprehensive quality assessment that can adapt to your specific business requirements.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
